<a href="https://colab.research.google.com/github/B1ackA1paca/cu-stip-ai-red/blob/main/stip_red.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets peft bitsandbytes accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [22]:
import os
import random
import pickle
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel
import torch.nn.functional as F
import random
import torch.optim as optim
import scipy.sparse as sp

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
os.chdir("/content/drive/MyDrive/cu-ai-blue")

In [5]:
device = ("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
def seed_everything(seed=42):
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.backends.cudnn.deterministic = True

seed_everything(42)

## data prepare

In [7]:
train = pd.read_json('kid_adult.jsonl', lines=True)
test = pd.read_json("public_test_style.jsonl", lines=True)

In [8]:
display(train.sample(1), test.sample(1))

,prompt,kid,adult
940,Что защищает наш мозг снаружи? — Мозг защищает...,Снаружи наш мозг защищает череп — это такая кр...,Снаружи центральную нервную систему надежно за...


,prompt,kid,adult,fact
49,Как дерево может превращаться в камень? За мил...,"Представь, что дерево — это пористая губка. Ес...",Процесс окаменения дерева (фоссилизации) проис...,Окаменелое дерево образуется при замещении орг...


In [9]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [10]:
class TrainDataset():
  def __init__(self, data):
    self.data = data
  def __len__(self):
    return len(self.data)
  def __getitem__(self, i):
    return self.data.loc[i, "prompt"], self.data.loc[i, "kid"]
class TestDataset():
  def __init__(self, data):
    self.data = data
  def __len__(self):
    return len(self.data)
  def __getitem__(self, i):
    return self.data.loc[i, "prompt"]

In [11]:
def collate_fn_train(batch):
  texts = []
  for i in batch:
    texts.append(f"<|im_start|>user\n{i[0]}<|im_end|>\n<|im_start|>assistant\n{i[1]}<|im_end|>")

  inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

  inputs["labels"] = inputs["input_ids"].clone()
  inputs["labels"][inputs["labels"] == tokenizer.pad_token_id] = -100

  return {
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs["attention_mask"],
    "labels": inputs["labels"]
  }
def collate_fn_test(batch):
  tokenizer.padding_side = "left"

  texts = []
  for i in batch:
    texts.append(f"<|im_start|>user\n{i}<|im_end|>\n<|im_start|>assistant\n")

  inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

  return {
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs["attention_mask"]
  }

In [12]:
train_part, val_part = train_test_split(train, test_size=0.2)
train_part, val_part = train_part.reset_index(drop=True), val_part.reset_index(drop=True)
train_loader = DataLoader(TrainDataset(train_part), batch_size=8, shuffle=True, collate_fn=collate_fn_train)
val_loader = DataLoader(TrainDataset(val_part), batch_size=8, shuffle=True, collate_fn=collate_fn_train)

## model

In [13]:
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_compute_dtype=torch.bfloat16,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
  "Qwen/Qwen3-4B-Instruct-2507",
  quantization_config=quantization_config,
  device_map="auto"
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
  r=16,
  lora_alpha=32,
  target_modules=["q_proj", "v_proj"],
  task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [14]:
optimizer = optim.AdamW(model.parameters(), lr=2e-4)

In [15]:
model.train()
epochs = 2

best_score, lag, max_lag = float("inf"), 0, 2
for epoch in range(epochs):
  model.train()
  for batch in tqdm(train_loader, desc=f"epoch: {epoch}"):
    optimizer.zero_grad()

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    loss = outputs.loss
    loss.backward()
    optimizer.step()
  model.eval()
  with torch.no_grad():
    sum_loss, k = 0, 0
    for batch in tqdm(val_loader, desc=f"val epoch: {epoch}"):
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)
      outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

      loss = outputs.loss.item()
      sum_loss += loss
      k += 1
    if (sum_loss / k) < best_score:
      best_score = (sum_loss / k)
      lag = 0
      model.save_pretrained("best_model")
    else:
      lag += 1
      if lag >= max_lag:
        break
model = PeftModel.from_pretrained(model, "best_model")

epoch: 0:   0%|          | 0/149 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
epoch: 1:  54%|█████▍    | 81/149 [23:17<19:05, 16.84s/it]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but

KeyboardInterrupt: 

In [23]:
test_dataset = TestDataset(test)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_test)

model.eval()
generated_texts = []

with torch.no_grad():
  for batch in tqdm(test_loader, desc="Generation"):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    outputs = model.generate(
      input_ids=input_ids,
      attention_mask=attention_mask,
      max_new_tokens=256,
      do_sample=False,
      pad_token_id=tokenizer.pad_token_id
    )

    for i in range(len(outputs)):
      prompt_len = input_ids[i].shape[0]
      gen_tokens = outputs[i][prompt_len:]
      gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
      generated_texts.append(gen_text)


with open("metrics/style_clf.pkl", "rb") as f:
  style_clf = pickle.load(f)

clf = style_clf["clf"]
vecs = style_clf["vecs"]

if len(vecs) == 1:
  X = vecs[0].transform(generated_texts)
else:
  X = sp.hstack([v.transform(generated_texts) for v in vecs])

probs = clf.predict_proba(X)
classes = list(clf.classes_)

if "kid" in classes:
  target_idx = classes.index("kid")
elif "simple" in classes:
  target_idx = classes.index("simple")
else:
  target_idx = 1

p_simple_values = probs[:, target_idx]
mean_p_simple = np.mean(p_simple_values)
print(mean_p_simple)

Generation: 100%|██████████| 4/4 [02:22<00:00, 35.74s/it]

0.9771807938862699


In [20]:
import pickle

with open("metrics/style_clf.pkl", "rb") as f:
  style_clf = pickle.load(f)

print({k: type(v) for k, v in style_clf.items()})

{'clf': <class 'sklearn.linear_model._logistic.LogisticRegression'>, 'vecs': <class 'tuple'>}


#taska 2

In [29]:
class DPODataset():
  def __init__(self, data):
    self.data = data.reset_index(drop=True)
  def __len__(self):
    return len(self.data)
  def __getitem__(self, i):
    return {
      "prompt": self.data.loc[i, "prompt"],
      "kid": self.data.loc[i, "kid"],
      "adult": self.data.loc[i, "adult"]
    }

def collate_fn_dpo(batch):
  texts = []
  for i in batch:
    texts.append((i["prompt"], i["kid"]))
  for i in batch:
    texts.append((i["prompt"], i["adult"]))

  full_batch = collate_fn_train(texts)
  B = len(batch)

  return {
    "chosen": {
      "input_ids": full_batch["input_ids"][:B],
      "attention_mask": full_batch["attention_mask"][:B],
      "labels": full_batch["labels"][:B]
    },
    "rejected": {
      "input_ids": full_batch["input_ids"][B:],
      "attention_mask": full_batch["attention_mask"][B:],
      "labels": full_batch["labels"][B:]
    }
  }

dpo_subset = train.sample(400, random_state=42)
dpo_dataset = DPODataset(dpo_subset)
dpo_loader = DataLoader(dpo_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn_dpo)

In [30]:
def get_log_probs(logits, labels):
  shift_logits = logits[..., :-1, :].contiguous()
  shift_labels = labels[..., 1:].contiguous()
  log_probs = F.log_softmax(shift_logits, dim=-1)
  loss_mask = (shift_labels != -100)
  dummy_labels = shift_labels.clone()
  dummy_labels[dummy_labels == -100] = 0
  per_token_logps = torch.gather(log_probs, dim=-1, index=dummy_labels.unsqueeze(-1)).squeeze(-1)
  return (per_token_logps * loss_mask).sum(dim=-1)

beta = 0.1
epochs = 1
optimizer = optim.AdamW(model.parameters(), lr=5e-6)

for epoch in range(epochs):
  model.train()
  train_loss, train_steps = 0, 0
  for batch in tqdm(dpo_loader, desc=f"DPO epoch: {epoch}"):
    optimizer.zero_grad()

    c = batch["chosen"]
    r = batch["rejected"]
    B = c["input_ids"].shape[0]

    input_ids = torch.cat([c["input_ids"], r["input_ids"]], dim=0).to(device)
    attention_mask = torch.cat([c["attention_mask"], r["attention_mask"]], dim=0).to(device)
    labels = torch.cat([c["labels"], r["labels"]], dim=0).to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    with torch.no_grad():
      with model.disable_adapter():
        ref_logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    all_logps = get_log_probs(logits, labels)
    all_ref_logps = get_log_probs(ref_logits, labels)

    policy_c_logps, policy_r_logps = all_logps[:B], all_logps[B:]
    ref_c_logps, ref_r_logps = all_ref_logps[:B], all_ref_logps[B:]

    policy_diff = policy_c_logps - policy_r_logps
    ref_diff = ref_c_logps - ref_r_logps

    loss = -F.logsigmoid(beta * (policy_diff - ref_diff)).mean()

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    train_steps += 1

  print(f"epoch: {epoch}, loss: {train_loss / train_steps}")

model.save_pretrained("best_dpo_style_model")

DPO epoch: 0: 100%|██████████| 100/100 [44:33<00:00, 26.73s/it]


epoch: 0, loss: 0.04864828290152218


In [31]:
model = PeftModel.from_pretrained(model, "best_dpo_style_model")

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.v_proj.

In [ ]:
import scipy.sparse as sp

test_dataset = TestDataset(test)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_test)

model.eval()
generated_texts = []

with torch.no_grad():
  for batch in tqdm(test_loader, desc="Generation"):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    outputs = model.generate(
      input_ids=input_ids,
      attention_mask=attention_mask,
      max_new_tokens=256,
      do_sample=False,
      pad_token_id=tokenizer.pad_token_id
    )

    for i in range(len(outputs)):
      prompt_len = input_ids[i].shape[0]
      gen_tokens = outputs[i][prompt_len:]
      gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
      generated_texts.append(gen_text)

with open("metrics/style_clf.pkl", "rb") as f:
  style_clf = pickle.load(f)

clf = style_clf["clf"]
vecs = style_clf["vecs"]

if len(vecs) == 1:
  X = vecs[0].transform(generated_texts)
else:
  X = sp.hstack([v.transform(generated_texts) for v in vecs])

probs = clf.predict_proba(X)
classes = list(clf.classes_)

if "kid" in classes:
  target_idx = classes.index("kid")
elif "simple" in classes:
  target_idx = classes.index("simple")
else:
  target_idx = 1

p_simple_values = probs[:, target_idx]
mean_p_simple = np.mean(p_simple_values)
print(mean_p_simple)

Generation:   0%|          | 0/4 [00:00<?, ?it/s]